# CineDub OSS — Production-Grade AI Dubbing

Open in Colab: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/cinedub/blob/main/cinedub.ipynb)

## Pipeline Overview
1. Download video → Normalize → Separate vocals → Diarize speakers → Transcribe → Align → Translate → Build speaker DB → Clone voices → Sync audio → Generate subtitles → Render final video → (optional) Lip-sync

In [ ]:
# Cell 00-01: Bootstrap + Drive Mount
import sys, os, subprocess

from google.colab import drive
drive.mount('/content/drive')

CINEDUB_ROOT = '/content/drive/MyDrive/cinedub'
os.makedirs(CINEDUB_ROOT, exist_ok=True)

# Copy our package into Colab (or pip install from repo)
if not os.path.isdir(f'{CINEDUB_ROOT}/cinedub'):
    !git clone --depth=1 https://github.com/your-org/cinedub {CINEDUB_ROOT}/repo
    !cp -r {CINEDUB_ROOT}/repo/cinedub {CINEDUB_ROOT}/

sys.path.insert(0, CINEDUB_ROOT)

DEPS = [
    'torch>=2.0', 'torchaudio', 'torchvision',
    'yt-dlp>=2026.3.17', 'whisperx>=3.8.5', 'pyannote.audio>=4.0.4',
    'transformers>=5.5', 'librosa', 'pydub', 'gradio>=5.0',
    'soundfile', 'accelerate', 'sentencepiece', 'protobuf',
]
for pkg in DEPS:
    !pip install -q $pkg

print('Bootstrap complete')

In [ ]:
# Cell 02-03: Runtime Detector + Config
import torch
from cinedub.core import Config, RuntimeMode, ModelManager

config = Config()
config.work_dir = CINEDUB_ROOT

mode = ModelManager.resolve_mode()
config.mode = mode

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {vram:.1f}GB')
else:
    print('CPU mode')
print(f'Runtime mode: {mode.value}')
config.ensure_dirs()

In [ ]:
# Cell 04-06: Project State + Checkpoint Manager
from cinedub.checkpoint import CheckpointManager
from cinedub.logging import CineDubLogger

log = CineDubLogger.get(config.log_dir)
checkpoint = CheckpointManager(config)

completed = checkpoint.completed_stages()
print(f'Completed stages: {completed}')
print(f'Log: {config.log_dir}')

In [ ]:
# Cell 07: Download Pipeline
from cinedub.stages import DownloadStage
from cinedub.core import PipelineContext

if config.source_url:
    ctx = PipelineContext(config=config)
    stage = DownloadStage()
    result = stage.execute(ctx)
    print(f'Download: {result.message}')
    if result.success:
        config.project_name = ctx.metadata.get('title', 'project')[:40]
        config.ensure_dirs()
        checkpoint.save('download', output_files=result.output_files)
else:
    print('Set config.source_url first (e.g., config.source_url = "https://...")')

In [ ]:
# Cell 08: Media Normalization
from cinedub.stages import NormalizeStage

if os.path.isfile(ctx.video_path):
    stage = NormalizeStage()
    result = stage.execute(ctx)
    print(f'Normalize: {result.message}')
    if result.success:
        checkpoint.save('normalize', output_files=result.output_files)
else:
    print('No video to normalize')

In [ ]:
# Cell 09: Audio Separation
from cinedub.stages import SeparationStage

stage = SeparationStage()
if stage.validate(ctx):
    result = stage.execute(ctx)
    print(f'Separation: {result.message}')
    if result.success:
        checkpoint.save('separation', output_files=result.output_files)
else:
    print('Skipping separation (CPU mode or no audio)')

In [ ]:
# Cell 10: Speaker Diarization
from cinedub.stages import DiarizationStage

stage = DiarizationStage()
if stage.validate(ctx):
    result = stage.execute(ctx)
    print(f'Diarization: {result.message}')
    if result.success:
        checkpoint.save('diarization', output_files=result.output_files)
else:
    print('Skipping diarization (CPU mode)')

In [ ]:
# Cell 11: WhisperX Transcription
from cinedub.stages import TranscriptionStage

stage = TranscriptionStage()
if stage.validate(ctx):
    result = stage.execute(ctx)
    print(f'Transcription: {result.message}')
    if result.success:
        checkpoint.save('transcription', output_files=result.output_files)
else:
    print('Skipping transcription (CPU mode)')

In [ ]:
# Cell 12: Forced Alignment
from cinedub.stages import AlignmentStage

stage = AlignmentStage()
result = stage.execute(ctx)
print(f'Alignment: {result.message}')
if result.success:
    checkpoint.save('alignment', output_files=result.output_files)

In [ ]:
# Cell 13: Translation (NLLB-200)
from cinedub.stages import TranslationStage

stage = TranslationStage()
result = stage.execute(ctx)
print(f'Translation: {result.message}')
if result.success:
    checkpoint.save('translation', output_files=result.output_files)

In [ ]:
# Cell 14: Speaker Database
from cinedub.stages import SpeakerDatabaseStage

stage = SpeakerDatabaseStage()
result = stage.execute(ctx)
print(f'Speaker DB: {result.message}')
if result.success:
    checkpoint.save('speaker_db', output_files=result.output_files)

In [ ]:
# Cell 15: GPT-SoVITS Voice Cloning
from cinedub.stages import VoiceCloningStage

stage = VoiceCloningStage()
if stage.validate(ctx):
    result = stage.execute(ctx)
    print(f'Voice cloning: {result.message}')
    if result.success:
        checkpoint.save('voice_cloning', output_files=result.output_files)
else:
    print('Skipping voice cloning (CPU mode)')

In [ ]:
# Cell 16: Audio Synchronization
from cinedub.stages import AudioSyncStage

stage = AudioSyncStage()
result = stage.execute(ctx)
print(f'Audio sync: {result.message}')
if result.success:
    checkpoint.save('audio_sync', output_files=result.output_files)

In [ ]:
# Cell 17: Subtitle Generation
from cinedub.stages import SubtitleStage

stage = SubtitleStage()
result = stage.execute(ctx)
print(f'Subtitles: {result.message}')
if result.success:
    checkpoint.save('subtitles', output_files=result.output_files)

In [ ]:
# Cell 18: Video Rendering (FFmpeg)
from cinedub.stages import RenderStage

stage = RenderStage()
result = stage.execute(ctx)
print(f'Render: {result.message}')
if result.success:
    checkpoint.save('render', output_files=result.output_files)
    print(f'Dubbed video: {ctx.metadata.get("dubbed_video_path", "")}')

In [ ]:
# Cell 19: Lip Sync (Optional)
from cinedub.stages import LipsyncStage

stage = LipsyncStage()
if stage.validate(ctx):
    result = stage.execute(ctx)
    print(f'Lip sync: {result.message}')
    if result.success:
        checkpoint.save('lipsync', output_files=result.output_files)
else:
    print('Skipping lip sync (disabled or insufficient GPU)')

In [ ]:
# Cell 20: Pipeline Orchestrator (run all stages sequentially)
from cinedub.stages.orchestrator import PipelineOrchestrator

config.source_url = config.source_url or input('YouTube URL: ')
config.target_language = config.target_language or input('Target language (es, fr, de, ...): ') or 'es'

orchestrator = PipelineOrchestrator(config)
results = orchestrator.run()

for r in results:
    status = '✅' if r.success else '❌'
    print(f'{status} {r.stage_name}: {r.message}')

final = orchestrator.ctx.metadata.get('dubbed_video_path', '')
print(f'\nFinal video: {final}')

In [ ]:
# Cell 21: Gradio UI
from cinedub.stages.ui import create_interface

demo = create_interface()
demo.launch(share=True, debug=True)

In [ ]:
# Cell 22: Recovery System
from cinedub.stages import RecoveryStage

stage = RecoveryStage()
result = stage.execute(ctx)
print(f'Recovery: {result.message}')

In [ ]:
# Cell 23: Cleanup
from cinedub.stages import CleanupStage

stage = CleanupStage()
result = stage.execute(ctx)
print(f'Cleanup: {result.message}')
print('\nDone! Check /content/drive/MyDrive/cinedub/projects/ for outputs.')